In [17]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from loguru import logger

project_root = Path.cwd().parent  # notebook eseguito da /notebooks
sys.path.insert(0, str(project_root / "src"))

from pipeline.orchestrator import run_pipeline

logger.remove()
logger.add(sys.stderr, level="ERROR")

print("python:", sys.executable)
print("pid:", os.getpid())
print("cwd:", Path.cwd())
print("project_root:", project_root)


python: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\.venv\Scripts\python.exe
pid: 9684
cwd: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\notebooks
project_root: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline


In [18]:
config_path = project_root / "configs" / "pipeline" / "pipeline_config_ALL_FEB.local.json"
print("config_path:", config_path)

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)

pd.DataFrame([
    {
        "extraction_config_path": config["extraction_config_path"],
        "year_ref": config["year_ref"],
        "dq_mutate": config["dataquality"]["mutate"],
        "percent_max_loss_ratio": config["cleaning"]["percent_max_loss_ratio"],
        "numeric_max_loss_ratio": config["cleaning"]["numeric_max_loss_ratio"],
    }
])


config_path: c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\configs\pipeline\pipeline_config_ALL_FEB.local.json


,extraction_config_path,year_ref,dq_mutate,percent_max_loss_ratio,numeric_max_loss_ratio
0,configs/extraction/extract_config_page_ALL_FEB...,2026,False,0.1,0.1


In [19]:
result = run_pipeline(config, project_root=project_root, publish=False)

qc_summary_df = pd.DataFrame(result["qc_summary"])
errors_df = pd.DataFrame(result["errors_list"])
log_percentage_df = pd.DataFrame(result["log_percentage"])
problem_tables = result["df_to_analyze"]

summary_df = pd.DataFrame([
    {
        "df_final_rows": int(result["df_final"].shape[0]),
        "df_final_cols": int(result["df_final"].shape[1]),
        "qc_rows": int(len(qc_summary_df)),
        "qc_problem_rows": int(qc_summary_df["is_problem"].sum()) if not qc_summary_df.empty else 0,
        "errors_rows": int(len(errors_df)),
        "df_to_analyze_count": int(len(problem_tables)),
        "log_percentage_rows": int(len(log_percentage_df)),
    }
])

summary_df


c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\src\pipeline\common\utilities.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[new_column_name] = df[column_lookup].where(mask).ffill()
c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\src\pipeline\common\utilities.py:71: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[new_column_name] = df[column_lookup].where(mask).ffill()
c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\src\pip

,df_final_rows,df_final_cols,qc_rows,qc_problem_rows,errors_rows,df_to_analyze_count,log_percentage_rows
0,1902,23,138,0,1,0,5


In [20]:
qc_summary_df.sort_values(["is_problem", "page", "idx"], ascending=[False, True, True])


,idx,page,n_col,is_problem,issues,dup_n,numeric_n,bad_chars_n,bad_tokens_n,dup_cols,numeric_cols,bad_chars_cols,bad_token_cols
0,0,2,15,False,,0,0,0,0,[],[],[],[]
1,1,2,15,False,,0,0,0,0,[],[],[],[]
2,2,3,18,False,,0,0,0,0,[],[],[],[]
3,3,3,18,False,,0,0,0,0,[],[],[],[]
4,4,5,18,False,,0,0,0,0,[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,133,74,18,False,,0,0,0,0,[],[],[],[]
134,134,75,18,False,,0,0,0,0,[],[],[],[]
135,135,75,18,False,,0,0,0,0,[],[],[],[]
136,136,76,18,False,,0,0,0,0,[],[],[],[]


In [21]:
errors_df


,page,n_col,e_des
0,28,16,Lunghezze non compatibili per rinomina colonne


In [22]:
log_percentage_df


,column,index,or_val
0,pct_vs_bdg,1265,%
1,pct_vs_bdg,1299,%
2,pct_vs_act,1263,by Business
3,pct_vs_act,1265,%
4,pct_vs_act,1299,%


In [23]:
problem_tables_overview = pd.DataFrame([
    {
        "idx": i,
        "page": int(df["page_num"].dropna().iloc[0]) if "page_num" in df.columns and not df["page_num"].dropna().empty else None,
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "sample_columns": ", ".join(map(str, list(df.columns[:8]))),
    }
    for i, df in enumerate(problem_tables)
])

problem_tables_overview


""


In [24]:
result["df_final"].head()


colonna_ordinata_post,page_num,capitolo,sottocapitolo,page_title,page_subtitle,page_subtitle_1,table_id,period_desc,period_start,item_agg_2,...,2026_act,2026_bdg,eur_vs_act,pct_vs_act,eur_vs_bdg,pct_vs_bdg,eur_delta,pct_delta,run_id,timestamp_utc
0,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,NaN,NaN,0,MONTH,2026-02-01,NaN,...,NaN,NaN,23054.0,0.068,11037.0,0.031,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630
1,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,NaN,NaN,0,MONTH,2026-02-01,NaN,...,NaN,NaN,8965.0,0.061,-1391.0,-0.009,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630
2,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,NaN,NaN,0,MONTH,2026-02-01,NaN,...,NaN,NaN,6377.0,0.059,4487.0,0.041,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630
3,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,NaN,NaN,0,MONTH,2026-02-01,NaN,...,NaN,NaN,5301.0,0.301,4968.0,0.277,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630
4,2,OVERVIEW,OVERVIEW,MENARINI GROUP PERFORMANCES OVERVIEW,NaN,NaN,0,MONTH,2026-02-01,NaN,...,NaN,NaN,2736.0,0.065,464.0,0.010,NaN,NaN,7321ef41-7075-4f9a-8b53-b071ee23ec5e,2026-03-30 12:26:12.668630


In [25]:
output_path = project_root / "data" / "output" / "df_final_FEB_2026.xlsx"

result["df_final"].to_excel(output_path, index=False)
print(output_path)


c:\work\MEN_Marketing\04_PRG_20251111___OCR_Sys_perlettura_PDF\PDFDataExtractor-pipeline\data\output\df_final_FEB_2026.xlsx
